# Retention Copilot: run the agent and the evaluation on Kaggle

**Before you start:** your code must be on GitHub (the repo is public). Settings (right panel): Accelerator = *GPU T4 x2* (or T4), *Internet = On*.

1. Edit `GITHUB_USER` in cell 1. 2. Run cells 1-4 and read the smoke-test output in cell 4. 3. Only if it looks sensible, run cell 5 (the full evaluation, roughly 20-40 minutes). 4. Run cell 6 and download `results.zip` from the Output panel.

If any cell errors, paste the error back.

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <- edit
MODEL = "granite4:micro"

# Install Ollama (Internet must be On)
!apt-get install -y -q zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the server in the background and download the model
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
!ollama pull {MODEL}

In [ ]:
# Get the code, install the extra packages, run the offline tests
%cd /kaggle/working
!rm -rf retention-copilot
!git clone https://github.com/{GITHUB_USER}/retention-copilot.git
%cd retention-copilot
!pip install -q langgraph langchain-ollama fastembed pytest
!python -m pytest -q 2>&1 | tail -5

In [ ]:
# Smoke test: 3 customers through the full agent, then 2 through the no-policy baseline. Read the drafts!
!python -m src.agent --customer auto --n 3 --config rag+verify --no-persist --model {MODEL}
!python -m src.agent --customer auto --n 2 --config none --no-persist --model {MODEL}

In [ ]:
# Full evaluation: 6 configurations x 48 held-out customers (resumable: re-run this cell if the session drops)
!python -m src.evaluate --model {MODEL} --n 48

In [ ]:
# Package the results and download results.zip from the Output panel (right side, /kaggle/working)
import zipfile, pathlib
root = pathlib.Path("/kaggle/working/retention-copilot")
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [root / "db" / "eval_runs.db", *root.glob("reports/**/*")]:
        if p.is_file():
            z.write(p, p.relative_to(root))
print(open(root / "reports" / "agent_eval.md").read())